# 1. Objetivo

Executar um conjunto pequeno e pré-especificado de **ANÁLISES EXPLORATÓRIAS SECUNDÁRIAS** envolvendo corticoide, atividade física, IMC e FINDRISC. Estas análises não substituem a hipótese principal, não selecionam resultados por p-valor e não permitem inferência causal.

# 2. Importações

In [1]:
from pathlib import Path
import hashlib
import platform
import sys

import numpy as np
import pandas as pd
import scipy
import statsmodels
from statsmodels.stats.multitest import multipletests

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cleaning import clean_numeric_column
from src.statistics import (
    RANDOM_STATE,
    descriptive_numeric,
    mann_whitney_with_bootstrap_ci,
    spearman_with_bootstrap_ci,
)
from src.variables import (
    AUTOIMMUNE_DIAGNOSES,
    count_autoimmune_diagnoses,
    create_atividade_fisica_atual,
    extract_findrisc_score,
)

# 3. Configuração

In [2]:
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'pacientes_clean.csv'
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'sociodemografico.csv'
TABLES_DIR = PROJECT_ROOT / 'outputs' / 'tables'
RESULTS_PATH = TABLES_DIR / 'secondary_analysis_results.csv'
DESCRIPTIVE_PATH = TABLES_DIR / 'secondary_analysis_descriptive.csv'
DECISIONS_PATH = TABLES_DIR / 'secondary_analysis_decisions.csv'
SENSITIVITY_RESULTS_PATH = TABLES_DIR / 'sensitivity_analysis_results.csv'
OUTLIER_AUDIT_PATH = TABLES_DIR / 'sensitivity_outlier_audit.csv'
DIAGNOSIS_RULE_PATH = TABLES_DIR / 'sensitivity_diagnosis_rule.csv'
N_BOOTSTRAP = 10_000
MIN_GROUP_SIZE = 10
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Python: {platform.python_version()}')
print(f'pandas: {pd.__version__}')
print(f'scipy: {scipy.__version__}')
print(f'statsmodels: {statsmodels.__version__}')
print(f'Bootstrap: {N_BOOTSTRAP:,} reamostragens; seed = {RANDOM_STATE}'.replace(',', '.'))

Python: 3.14.6
pandas: 3.0.5
scipy: 1.18.0
statsmodels: 0.14.6
Bootstrap: 10.000 reamostragens; seed = 42


# 4. Carregamento

Somente variáveis agregáveis necessárias para as análises e para a avaliação de suficiência são carregadas. Nenhum identificador ou dado nominal entra no notebook.

In [3]:
columns = [
    'imc', 'findrisc_score', 'uso_corticoide', 'atividade_fisica',
    'uso_imunobiologico', 'tabagismo', 'etilismo',
    'renda_mensal_familiar', 'escolaridade', 'tempo_diagnostico',
    'diagnostico_padronizado',
]
df = pd.read_csv(PROCESSED_PATH, usecols=columns)
df['uso_corticoide'] = df['uso_corticoide'].astype('boolean')
df['atividade_fisica_atual'] = create_atividade_fisica_atual(df['atividade_fisica'])
assert len(df) == 75
print(f'Amostra disponível: {len(df)} pacientes')
print('Variáveis carregadas sem identificação individual.')

Amostra disponível: 75 pacientes
Variáveis carregadas sem identificação individual.


# 5. Validações e adequação

A atividade física atual foi derivada por regra explícita: `Não`, `Atualmente não` e relatos iniciados por `Parou` foram classificados como ausência atual; os demais relatos válidos descrevem atividade atual. Missing permanece missing.

Para os quatro contrastes planejados, há dois grupos independentes com pelo menos 10 observações válidas por desfecho. O Mann–Whitney bilateral foi escolhido para comparar distribuições sem exigir normalidade, especialmente porque FINDRISC é discreto e limitado. O efeito é a correlação bisserial de postos, orientada como `Sim − Não`, com IC95% bootstrap percentil.

In [4]:
assert df['uso_corticoide'].notna().all()
assert df['atividade_fisica_atual'].notna().all()
assert df['uso_corticoide'].value_counts().to_dict() == {False: 45, True: 30}
assert df['atividade_fisica_atual'].value_counts().to_dict() == {False: 42, True: 33}
assert pd.api.types.is_numeric_dtype(df['imc'])
assert pd.api.types.is_numeric_dtype(df['findrisc_score'])
print('Validação dos grupos planejados:')
print('- Corticoide: Não = 45; Sim = 30; missing = 0')
print('- Atividade física atual: Não = 42; Sim = 33; missing = 0')
print('- Todos os grupos planejados superam o mínimo de 10 observações por desfecho.')

Validação dos grupos planejados:
- Corticoide: Não = 45; Sim = 30; missing = 0
- Atividade física atual: Não = 42; Sim = 33; missing = 0
- Todos os grupos planejados superam o mínimo de 10 observações por desfecho.


In [5]:
decisions = pd.DataFrame([
    {
        'variavel': 'Corticoide',
        'status': 'Realizada',
        'analises': 'IMC; FINDRISC',
        'justificativa': 'Campo binário completo; grupos com 45 e 30 pacientes.',
    },
    {
        'variavel': 'Atividade física',
        'status': 'Realizada',
        'analises': 'IMC; FINDRISC',
        'justificativa': 'Status atual derivado por regra explícita; grupos com 42 e 33 pacientes.',
    },
    {
        'variavel': 'Imunobiológico',
        'status': 'Não realizada',
        'analises': pd.NA,
        'justificativa': '63 valores presentes, 12 ausentes (16,0%) e 29 categorias textuais semanticamente inconsistentes; não há agrupamento clínico binário confiável sem adjudicação.',
    },
    {
        'variavel': 'Tabagismo',
        'status': 'Não realizada',
        'analises': pd.NA,
        'justificativa': '17 categorias textuais misturam nunca fumou, exposição passiva, uso atual e cessação; grupos atuais são muito pequenos.',
    },
    {
        'variavel': 'Etilismo',
        'status': 'Não realizada',
        'analises': pd.NA,
        'justificativa': '25 categorias textuais sem separação padronizada entre uso atual, frequência e uso passado.',
    },
    {
        'variavel': 'Renda',
        'status': 'Não realizada',
        'analises': pd.NA,
        'justificativa': 'Quatro estratos; o menor contém apenas 4 pacientes, insuficiente para comparação multigrupo estável.',
    },
    {
        'variavel': 'Escolaridade',
        'status': 'Não realizada',
        'analises': pd.NA,
        'justificativa': 'Sete estratos, incluindo grupos com 1 e 3 pacientes; comparação multigrupo seria instável.',
    },
    {
        'variavel': 'Tempo de doença',
        'status': 'Não realizada',
        'analises': pd.NA,
        'justificativa': '73 valores presentes em 55 formatos que misturam duração, ano-calendário e idade ao diagnóstico; unidade não padronizada.',
    },
])
decisions.to_csv(DECISIONS_PATH, index=False)
print('Decisões metodológicas: 2 exposições analisadas e 6 variáveis não testadas.')

Decisões metodológicas: 2 exposições analisadas e 6 variáveis não testadas.


# 6. Análise

Os quatro testes abaixo foram definidos antes de examinar seus p-valores. Para cada teste, o N e as estatísticas dos dois grupos são calculados primeiro; o teste só prossegue se ambos os grupos tiverem ao menos 10 observações válidas.

In [6]:
specifications = [
    ('Corticoide', 'uso_corticoide', 'IMC', 'imc'),
    ('Corticoide', 'uso_corticoide', 'FINDRISC', 'findrisc_score'),
    ('Atividade física atual', 'atividade_fisica_atual', 'IMC', 'imc'),
    ('Atividade física atual', 'atividade_fisica_atual', 'FINDRISC', 'findrisc_score'),
]
result_rows = []
descriptive_rows = []

for exposure_label, exposure_column, outcome_label, outcome_column in specifications:
    group = df[exposure_column]
    reference = df.loc[group.eq(False), outcome_column]
    comparison = df.loc[group.eq(True), outcome_column]
    reference_summary = descriptive_numeric(reference)
    comparison_summary = descriptive_numeric(comparison)
    n_reference = reference_summary['n_valido']
    n_comparison = comparison_summary['n_valido']

    print(f'ANÁLISE EXPLORATÓRIA SECUNDÁRIA — {exposure_label} × {outcome_label}')
    print(f'N total = {n_reference + n_comparison}; Não = {n_reference}; Sim = {n_comparison}')
    for group_label, summary in [('Não', reference_summary), ('Sim', comparison_summary)]:
        description = (
            f"{group_label}: média {summary['media']:.2f} ± {summary['desvio_padrao']:.2f}; "
            f"mediana {summary['mediana']:.2f} [{summary['p25']:.2f}–{summary['p75']:.2f}]"
        ).replace('.', ',')
        print(description)
        descriptive_rows.append({
            'analise': 'ANÁLISE EXPLORATÓRIA SECUNDÁRIA',
            'exposicao': exposure_label,
            'desfecho': outcome_label,
            'grupo': group_label,
            'n_valido': summary['n_valido'],
            'n_ausente_no_grupo': summary['n_ausente'],
            'media': summary['media'],
            'desvio_padrao': summary['desvio_padrao'],
            'mediana': summary['mediana'],
            'p25': summary['p25'],
            'p75': summary['p75'],
            'minimo': summary['minimo'],
            'maximo': summary['maximo'],
        })

    assert min(n_reference, n_comparison) >= MIN_GROUP_SIZE
    adequacy = (
        'Adequado para comparação exploratória de duas amostras independentes; '
        'Mann–Whitney compara distribuições e não pressupõe normalidade.'
    )
    print(f'Adequação: {adequacy}')
    test_result = mann_whitney_with_bootstrap_ci(
        reference,
        comparison,
        n_bootstrap=N_BOOTSTRAP,
        random_state=RANDOM_STATE,
    )
    result_rows.append({
        'analise': 'ANÁLISE EXPLORATÓRIA SECUNDÁRIA',
        'exposicao': exposure_label,
        'desfecho': outcome_label,
        'grupo_referencia': 'Não',
        'grupo_comparacao': 'Sim',
        'n_total': n_reference + n_comparison,
        'n_referencia': test_result['n_referencia'],
        'n_comparacao': test_result['n_comparacao'],
        'teste': 'Mann–Whitney bilateral',
        'u': test_result['u'],
        'tamanho_efeito': 'Correlação bisserial de postos',
        'efeito': test_result['correlacao_bisserial_postos'],
        'ic95_inferior': test_result['ic95_inferior'],
        'ic95_superior': test_result['ic95_superior'],
        'p_valor': test_result['p'],
        'bootstrap': test_result['bootstrap'],
        'random_state': test_result['random_state'],
        'adequacao': adequacy,
    })
    print()

ANÁLISE EXPLORATÓRIA SECUNDÁRIA — Corticoide × IMC
N total = 73; Não = 44; Sim = 29
Não: média 28,30 ± 4,87; mediana 27,44 [24,35–31,40]
Sim: média 29,37 ± 6,38; mediana 28,91 [24,79–33,43]
Adequação: Adequado para comparação exploratória de duas amostras independentes; Mann–Whitney compara distribuições e não pressupõe normalidade.

ANÁLISE EXPLORATÓRIA SECUNDÁRIA — Corticoide × FINDRISC
N total = 74; Não = 44; Sim = 30
Não: média 13,66 ± 6,24; mediana 14,00 [8,75–19,00]
Sim: média 13,90 ± 6,39; mediana 13,50 [9,00–20,00]
Adequação: Adequado para comparação exploratória de duas amostras independentes; Mann–Whitney compara distribuições e não pressupõe normalidade.

ANÁLISE EXPLORATÓRIA SECUNDÁRIA — Atividade física atual × IMC
N total = 73; Não = 40; Sim = 33
Não: média 29,11 ± 5,67; mediana 27,80 [25,46–32,62]
Sim: média 28,25 ± 5,34; mediana 27,94 [24,20–31,20]
Adequação: Adequado para comparação exploratória de duas amostras independentes; Mann–Whitney compara distribuições e não p

In [7]:
results = pd.DataFrame(result_rows)
descriptive = pd.DataFrame(descriptive_rows)
results['p_ajustado_holm'] = multipletests(
    results['p_valor'], alpha=0.05, method='holm'
)[1]
result_columns = [
    'analise', 'exposicao', 'desfecho', 'grupo_referencia', 'grupo_comparacao',
    'n_total', 'n_referencia', 'n_comparacao', 'teste', 'u', 'tamanho_efeito',
    'efeito', 'ic95_inferior', 'ic95_superior', 'p_valor', 'p_ajustado_holm',
    'bootstrap', 'random_state', 'adequacao',
]
results = results[result_columns]
results.to_csv(RESULTS_PATH, index=False)
descriptive.to_csv(DESCRIPTIVE_PATH, index=False)

print('Ajuste de multiplicidade: método de Holm para 4 testes.')
for row in results.itertuples(index=False):
    print(f'ANÁLISE EXPLORATÓRIA SECUNDÁRIA — {row.exposicao} × {row.desfecho}')
    print('Teste: Mann–Whitney bilateral')
    print('Tamanho de efeito: correlação bisserial de postos')
    print(f'U = {row.u:.1f}')
    print(f'Efeito = {row.efeito:.3f}'.replace('.', ','))
    print(
        f'IC95% bootstrap = {row.ic95_inferior:.3f} a {row.ic95_superior:.3f}'.replace('.', ',')
    )
    print(f'p bruto = {row.p_valor:.4f}; p Holm = {row.p_ajustado_holm:.4f}'.replace('.', ','))
    print()
results

Ajuste de multiplicidade: método de Holm para 4 testes.
ANÁLISE EXPLORATÓRIA SECUNDÁRIA — Corticoide × IMC
Teste: Mann–Whitney bilateral
Tamanho de efeito: correlação bisserial de postos
U = 715.0
Efeito = 0,121
IC95% bootstrap = -0,161 a 0,395
p bruto = 0,3884; p Holm = 1,0000

ANÁLISE EXPLORATÓRIA SECUNDÁRIA — Corticoide × FINDRISC
Teste: Mann–Whitney bilateral
Tamanho de efeito: correlação bisserial de postos
U = 668.0
Efeito = 0,012
IC95% bootstrap = -0,264 a 0,288
p bruto = 0,9340; p Holm = 1,0000

ANÁLISE EXPLORATÓRIA SECUNDÁRIA — Atividade física atual × IMC
Teste: Mann–Whitney bilateral
Tamanho de efeito: correlação bisserial de postos
U = 611.0
Efeito = -0,074
IC95% bootstrap = -0,342 a 0,198
p bruto = 0,5909; p Holm = 1,0000

ANÁLISE EXPLORATÓRIA SECUNDÁRIA — Atividade física atual × FINDRISC
Teste: Mann–Whitney bilateral
Tamanho de efeito: correlação bisserial de postos
U = 499.5
Efeito = -0,257
IC95% bootstrap = -0,514 a 0,007
p bruto = 0,0600; p Holm = 0,2399



,analise,exposicao,desfecho,grupo_referencia,grupo_comparacao,n_total,n_referencia,n_comparacao,teste,u,tamanho_efeito,efeito,ic95_inferior,ic95_superior,p_valor,p_ajustado_holm,bootstrap,random_state,adequacao
0,ANÁLISE EXPLORATÓRIA SECUNDÁRIA,Corticoide,IMC,Não,Sim,73,44,29,Mann–Whitney bilateral,715.0,Correlação bisserial de postos,0.120690,-0.161442,0.395004,0.388429,1.000000,10000,42,Adequado para comparação exploratória de duas ...
1,ANÁLISE EXPLORATÓRIA SECUNDÁRIA,Corticoide,FINDRISC,Não,Sim,74,44,30,Mann–Whitney bilateral,668.0,Correlação bisserial de postos,0.012121,-0.263636,0.287879,0.934041,1.000000,10000,42,Adequado para comparação exploratória de duas ...
2,ANÁLISE EXPLORATÓRIA SECUNDÁRIA,Atividade física atual,IMC,Não,Sim,73,40,33,Mann–Whitney bilateral,611.0,Correlação bisserial de postos,-0.074242,-0.342443,0.197727,0.590850,1.000000,10000,42,Adequado para comparação exploratória de duas ...
3,ANÁLISE EXPLORATÓRIA SECUNDÁRIA,Atividade física atual,FINDRISC,Não,Sim,74,42,32,Mann–Whitney bilateral,499.5,Correlação bisserial de postos,-0.256696,-0.514137,0.007459,0.059974,0.239897,10000,42,Adequado para comparação exploratória de duas ...


## 6.1 Análises de sensibilidade

A análise principal é repetida na subamostra com exatamente um diagnóstico autoimune explícito. A contagem considera componentes diagnósticos separados por vírgula; condições não autoimunes concomitantes não excluem o paciente. Termos incertos não são inventados como autoimunes.

Os extremos de IMC e FINDRISC também são confrontados com o dado bruto, a transformação, faixas objetivas e o critério de 1,5×IQR. Nenhum resultado é selecionado pelo menor p-valor.

In [8]:
diagnosis_classification = {
    'Arterite temporal': 'Autoimune',
    'Artrite reumatoide': 'Autoimune',
    'Espondiloartrite': 'Autoimune',
    'Lúpus eritematoso': 'Autoimune',
    'Síndrome antifosfolipídica (SAF)': 'Autoimune',
    'Artrose': 'Não autoimune',
    'Fibromialgia': 'Não autoimune',
    'Osteoporose': 'Não autoimune',
    'Doença de BC': 'Incerto',
    'Em descoberta': 'Incerto',
}
observed_terms = sorted({
    component.strip()
    for diagnosis in df['diagnostico_padronizado'].dropna()
    for component in diagnosis.split(',')
})
assert set(diagnosis_classification) == set(observed_terms)
assert {term for term, status in diagnosis_classification.items() if status == 'Autoimune'} == set(
    AUTOIMMUNE_DIAGNOSES
)
diagnosis_rule = pd.DataFrame(
    [{'termo': term, 'classificacao': diagnosis_classification[term]} for term in observed_terms]
)
diagnosis_rule.to_csv(DIAGNOSIS_RULE_PATH, index=False)

n_autoimmune = count_autoimmune_diagnoses(df['diagnostico_padronizado'])
assert n_autoimmune.value_counts().to_dict() == {1: 68, 0: 6, 2: 1}
single_autoimmune = n_autoimmune.eq(1)
print('Regra diagnóstica explícita:')
print('- Exatamente um diagnóstico autoimune: 68 pacientes')
print('- Nenhum diagnóstico autoimune explícito: 6 pacientes')
print('- Mais de um diagnóstico autoimune: 1 paciente')

Regra diagnóstica explícita:
- Exatamente um diagnóstico autoimune: 68 pacientes
- Nenhum diagnóstico autoimune explícito: 6 pacientes
- Mais de um diagnóstico autoimune: 1 paciente


In [9]:
all_pairs = df[['imc', 'findrisc_score']].dropna()
single_pairs = df.loc[single_autoimmune, ['imc', 'findrisc_score']].dropna()
sensitivity_rows = []
for scenario, eligible_n, data in [
    ('Todos os pacientes com pares válidos', len(df), all_pairs),
    ('Apenas um diagnóstico autoimune', int(single_autoimmune.sum()), single_pairs),
]:
    estimate = spearman_with_bootstrap_ci(
        data['imc'],
        data['findrisc_score'],
        n_bootstrap=N_BOOTSTRAP,
        random_state=RANDOM_STATE,
    )
    sensitivity_rows.append({
        'cenario': scenario,
        'n_elegivel': eligible_n,
        'n': estimate['n'],
        'rho': estimate['rho'],
        'ic95_inferior': estimate['ic95_inferior'],
        'ic95_superior': estimate['ic95_superior'],
        'p_valor': estimate['p'],
        'metodo_ic': 'Bootstrap percentil pareado',
        'bootstrap': estimate['bootstrap'],
        'random_state': estimate['random_state'],
    })
sensitivity_results = pd.DataFrame(sensitivity_rows)
sensitivity_results.to_csv(SENSITIVITY_RESULTS_PATH, index=False)
main_sensitivity = sensitivity_results.iloc[0]
single_sensitivity = sensitivity_results.iloc[1]
rho_difference = abs(single_sensitivity['rho'] - main_sensitivity['rho'])

print('ANÁLISE DE SENSIBILIDADE — DIAGNÓSTICOS MÚLTIPLOS')
print(
    f"Todos os pacientes: N = {int(main_sensitivity['n'])}; "
    f"rho = {main_sensitivity['rho']:.3f}; "
    f"IC95% {main_sensitivity['ic95_inferior']:.3f} a {main_sensitivity['ic95_superior']:.3f}; "
    f"p = {main_sensitivity['p_valor']:.3e}".replace('.', ',')
)
print(
    f"Um diagnóstico autoimune: N = {int(single_sensitivity['n'])}; "
    f"rho = {single_sensitivity['rho']:.3f}; "
    f"IC95% {single_sensitivity['ic95_inferior']:.3f} a {single_sensitivity['ic95_superior']:.3f}; "
    f"p = {single_sensitivity['p_valor']:.3e}".replace('.', ',')
)
print(f'Diferença absoluta entre os rhos: {rho_difference:.3f}'.replace('.', ','))
print('A conclusão principal permaneceu aproximadamente estável.')
print('O cenário de sensibilidade não foi escolhido por apresentar menor p-valor.')
sensitivity_results

ANÁLISE DE SENSIBILIDADE — DIAGNÓSTICOS MÚLTIPLOS
Todos os pacientes: N = 72; rho = 0,704; IC95% 0,561 a 0,808; p = 5,027e-12
Um diagnóstico autoimune: N = 65; rho = 0,764; IC95% 0,643 a 0,843; p = 1,395e-13
Diferença absoluta entre os rhos: 0,059
A conclusão principal permaneceu aproximadamente estável.
O cenário de sensibilidade não foi escolhido por apresentar menor p-valor.


,cenario,n_elegivel,n,rho,ic95_inferior,ic95_superior,p_valor,metodo_ic,bootstrap,random_state
0,Todos os pacientes com pares válidos,75,72,0.704457,0.561385,0.807711,5.026910e-12,Bootstrap percentil pareado,10000,42
1,Apenas um diagnóstico autoimune,68,65,0.763596,0.643389,0.843178,1.395474e-13,Bootstrap percentil pareado,10000,42


In [10]:
raw_extremes = pd.read_csv(
    RAW_PATH,
    usecols=['IMC', 'resultado do findrisk'],
    dtype='string',
)
raw_imc = clean_numeric_column(raw_extremes['IMC']).dropna()
raw_findrisc = extract_findrisc_score(raw_extremes['resultado do findrisk']).dropna()

def audit_extremes(variable, processed, raw_values, plausible_minimum, plausible_maximum):
    valid = processed.dropna()
    q1, q3 = valid.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    rows = []
    for extreme, value, raw_value in [
        ('Mínimo', float(valid.min()), float(raw_values.min())),
        ('Máximo', float(valid.max()), float(raw_values.max())),
    ]:
        present_in_raw = bool(np.isclose(raw_values.astype(float), value).any())
        transformation_consistent = bool(np.isclose(value, raw_value))
        plausible = plausible_minimum <= value <= plausible_maximum
        iqr_outlier = not lower_fence <= value <= upper_fence
        if not present_in_raw or not transformation_consistent:
            classification = 'Erro de transformação'
        elif not plausible:
            classification = 'Biologicamente implausível'
        else:
            classification = 'Valor extremo verdadeiro'
        rows.append({
            'variavel': variable,
            'extremo': extreme,
            'valor': value,
            'presente_no_bruto': present_in_raw,
            'transformacao_consistente': transformation_consistent,
            'faixa_plausivel': plausible,
            'limite_iqr_inferior': lower_fence,
            'limite_iqr_superior': upper_fence,
            'outlier_iqr': iqr_outlier,
            'classificacao': classification,
            'excluido': False,
            'justificativa': (
                'Valor confirmado no bruto, transformação consistente e faixa válida; mantido.'
                if classification == 'Valor extremo verdadeiro'
                else 'Requer adjudicação da fonte antes de qualquer exclusão.'
            ),
        })
    return rows

outlier_audit = pd.DataFrame(
    audit_extremes('IMC', df['imc'], raw_imc, 10, 80)
    + audit_extremes('FINDRISC', df['findrisc_score'], raw_findrisc, 0, 26)
)
outlier_audit.to_csv(OUTLIER_AUDIT_PATH, index=False)
assert outlier_audit['classificacao'].eq('Valor extremo verdadeiro').all()
assert not outlier_audit['outlier_iqr'].any()
assert not outlier_audit['excluido'].any()
print('ANÁLISE DE SENSIBILIDADE — EXTREMOS')
print('Erro de entrada identificado: 0')
print('Erro de transformação identificado: 0')
print('Valor biologicamente implausível identificado: 0')
print('Extremos observados confirmados como valores verdadeiros: 4')
print('Outliers pelo critério de 1,5×IQR: 0')
print('Nenhum valor foi excluído após a investigação de extremos.')
outlier_audit

ANÁLISE DE SENSIBILIDADE — EXTREMOS
Erro de entrada identificado: 0
Erro de transformação identificado: 0
Valor biologicamente implausível identificado: 0
Extremos observados confirmados como valores verdadeiros: 4
Outliers pelo critério de 1,5×IQR: 0
Nenhum valor foi excluído após a investigação de extremos.


,variavel,extremo,valor,presente_no_bruto,transformacao_consistente,faixa_plausivel,limite_iqr_inferior,limite_iqr_superior,outlier_iqr,classificacao,excluido,justificativa
0,IMC,Mínimo,17.5,True,True,True,13.1,43.82,False,Valor extremo verdadeiro,False,"Valor confirmado no bruto, transformação consi..."
1,IMC,Máximo,41.1,True,True,True,13.1,43.82,False,Valor extremo verdadeiro,False,"Valor confirmado no bruto, transformação consi..."
2,FINDRISC,Mínimo,2.0,True,True,True,-6.0,34.00,False,Valor extremo verdadeiro,False,"Valor confirmado no bruto, transformação consi..."
3,FINDRISC,Máximo,24.0,True,True,True,-6.0,34.00,False,Valor extremo verdadeiro,False,"Valor confirmado no bruto, transformação consi..."


### Conclusão da sensibilidade

A direção positiva, a magnitude aproximada e os intervalos de confiança sobrepostos sustentam estabilidade aproximada da conclusão principal na subamostra com um diagnóstico autoimune. Essa estabilidade não elimina a limitação estrutural: o IMC compõe o FINDRISC, de modo que parte da associação continua esperada pela construção do escore. Nenhum extremo verdadeiro foi excluído.

# 7. Visualização

Não foram adicionadas figuras secundárias. A decisão evita multiplicar outputs de baixa prioridade e preserva as poucas figuras principais definidas no projeto. As distribuições de cada grupo permanecem documentadas numericamente nas tabelas exportadas.

# 8. Conclusões deste notebook

As quatro comparações são exploratórias. A interpretação prioriza tamanho de efeito e IC95%, considera o ajuste de Holm e não transforma resultados imprecisos em prova de ausência de associação.

A atividade física é um componente do próprio FINDRISC. Portanto, qualquer contraste entre atividade física e FINDRISC também possui acoplamento matemático parcial, além das limitações de autorrelato e do desenho transversal.

In [11]:
corticoid_results = results[results['exposicao'].eq('Corticoide')]
activity_imc = results[
    results['exposicao'].eq('Atividade física atual') & results['desfecho'].eq('IMC')
].iloc[0]
activity_findrisc = results[
    results['exposicao'].eq('Atividade física atual') & results['desfecho'].eq('FINDRISC')
].iloc[0]

print(
    'Corticoide: os efeitos estimados para IMC e FINDRISC foram pequenos, com IC95% '
    'compatíveis com efeitos em ambas as direções.'
)
print(
    'Atividade física × IMC: efeito pequeno e impreciso, com IC95% incluindo o valor nulo.'
)
print(
    'Atividade física × FINDRISC: pacientes ativos apresentaram postos menores, mas o IC95% '
    'incluiu o valor nulo e p ajustado de Holm foi 0,240.'
)
print(
    'A atividade física integra o cálculo do FINDRISC; parte desse contraste é esperada pela '
    'construção matemática do escore.'
)
print('Nenhuma interpretação causal foi realizada.')
print('Variáveis não testadas e respectivos motivos estão documentados na tabela de decisões.')

Corticoide: os efeitos estimados para IMC e FINDRISC foram pequenos, com IC95% compatíveis com efeitos em ambas as direções.
Atividade física × IMC: efeito pequeno e impreciso, com IC95% incluindo o valor nulo.
Atividade física × FINDRISC: pacientes ativos apresentaram postos menores, mas o IC95% incluiu o valor nulo e p ajustado de Holm foi 0,240.
A atividade física integra o cálculo do FINDRISC; parte desse contraste é esperada pela construção matemática do escore.
Nenhuma interpretação causal foi realizada.
Variáveis não testadas e respectivos motivos estão documentados na tabela de decisões.


# 9. Outputs gerados

Os outputs contêm apenas resultados agregados. O arquivo bruto não é alterado.

In [12]:
output_paths = [
    RESULTS_PATH, DESCRIPTIVE_PATH, DECISIONS_PATH,
    SENSITIVITY_RESULTS_PATH, OUTLIER_AUDIT_PATH, DIAGNOSIS_RULE_PATH,
]
assert all(path.exists() and path.stat().st_size > 0 for path in output_paths)
assert len(results) == 4 and len(descriptive) == 8 and len(decisions) == 8
assert len(sensitivity_results) == 2 and len(outlier_audit) == 4
assert hashlib.sha256(RAW_PATH.read_bytes()).hexdigest() == (
    'a895b5340c856d2cd1772c8e115ed5efc20f409d3aef5ecba14da23d44da9bf4'
)
print('Outputs agregados gerados:')
for path in output_paths:
    print(f'- {path.relative_to(PROJECT_ROOT)}')
print('Integridade do arquivo bruto confirmada por SHA-256.')

Outputs agregados gerados:
- outputs/tables/secondary_analysis_results.csv
- outputs/tables/secondary_analysis_descriptive.csv
- outputs/tables/secondary_analysis_decisions.csv
- outputs/tables/sensitivity_analysis_results.csv
- outputs/tables/sensitivity_outlier_audit.csv
- outputs/tables/sensitivity_diagnosis_rule.csv
Integridade do arquivo bruto confirmada por SHA-256.
